In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (13, 6)

Part 1
Data Acquisition and Cleaning

In [26]:
import os
import sqlite3
from pathlib import Path

repo_root = Path('/workspaces/codespaces-jupyter')
excel_candidates = [
    repo_root / 'data' / 'DatasetEvidence.xlsx',
    repo_root / 'DatasetEvidence.xlsx',
    Path.cwd() / 'data' / 'DatasetEvidence.xlsx',
    Path.cwd() / 'DatasetEvidence.xlsx',
]
excel_path = next((p for p in excel_candidates if p.exists()), excel_candidates[0])

fondo = pd.read_excel(
    excel_path,
    sheet_name='Stocks',
    header=1,
    index_col='Exchange Date',
    parse_dates=True
)

data_prices = fondo.sort_index(ascending=True)
data_prices = data_prices.ffill().bfill().dropna()

fondo_weekly_returns = data_prices.pct_change().dropna()

connection = sqlite3.connect('fondo.db')
data_prices.to_sql('fondo', connection, if_exists='replace', index=True)
fondo_weekly_returns.to_sql('Fondo_Weekly_Returns', connection, if_exists='replace', index=True)
connection.close()

print("Prices")
display(data_prices.head())

print("Weekly Returns")
display(fondo_weekly_returns.head())

Prices


,S&P,IPC,AAPL,MSFT,GOOGL,AMZN,NVDA,WALMEX,FEMSA,CEMEX,AC
Exchange Date,,,,,,,,,,,
2021-09-17,4432.99,2563.578177,146.06,299.87,140.8000,173.126,21.900,3.623945,8.707861,0.729486,6.180143
2021-09-24,4455.48,2550.185130,146.92,299.35,142.2150,171.276,22.081,3.565369,8.873253,0.738024,6.145210
2021-10-01,4357.04,2499.390572,142.65,289.10,136.5430,164.163,20.742,3.434334,8.615693,0.718097,6.063439
2021-10-08,4391.34,2471.681570,142.90,294.85,139.7855,164.431,20.831,3.402291,8.313113,0.657837,6.052008
2021-10-15,4471.37,2597.195140,144.84,304.21,141.3680,170.451,21.862,3.495991,8.437700,0.724089,6.228541


Weekly Returns


,S&P,IPC,AAPL,MSFT,GOOGL,AMZN,NVDA,WALMEX,FEMSA,CEMEX,AC
Exchange Date,,,,,,,,,,,
2021-09-24,0.005073,-0.005224,0.005888,-0.001734,0.010050,-0.010686,0.008265,-0.016164,0.018993,0.011704,-0.005653
2021-10-01,-0.022094,-0.019918,-0.029063,-0.034241,-0.039883,-0.041529,-0.060640,-0.036752,-0.029027,-0.027001,-0.013306
2021-10-08,0.007872,-0.011086,0.001753,0.019889,0.023747,0.001633,0.004291,-0.009330,-0.035120,-0.083915,-0.001885
2021-10-15,0.018225,0.050781,0.013576,0.031745,0.011321,0.036611,0.049494,0.027540,0.014987,0.100711,0.029169
2021-10-22,0.016445,-0.009218,0.026581,0.016272,-0.026891,-0.021552,0.039521,0.043596,-0.005385,-0.067203,-0.006676


part 2

In [28]:
stocks = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'WALMEX', 'FEMSA', 'CEMEX', 'AC']

data_weekly_returns = data_prices.pct_change().dropna()

us_market_returns = data_weekly_returns['S&P']
mx_market_returns = data_weekly_returns['IPC']

market_correlation = data_weekly_returns['S&P'].rolling(12).corr(data_weekly_returns['IPC'])
volatility = data_weekly_returns.std()

connection = sqlite3.connect('fondo.db')
data_weekly_returns.to_sql('Fondo_Weekly_Returns', connection, if_exists='replace', index=True)
volatility.to_sql('Fondo_Volatility', connection, if_exists='replace', index=True)

data_market_features = pd.DataFrame({
    'US_Market_Returns': us_market_returns,
    'MX_Market_Returns': mx_market_returns,
    'Market_Correlation': market_correlation,
}, index=data_weekly_returns.index)

data_market_features.to_sql('Fondo_Market_Features', connection, if_exists='replace', index=True)

connection.close()

print("Dependent Variable : Weekly Returns")
display(data_weekly_returns[stocks].head())

print("Independent Variables : Market Features")
display(data_market_features.head())
display(volatility.head())

Dependent Variable : Weekly Returns


,AAPL,MSFT,GOOGL,AMZN,NVDA,WALMEX,FEMSA,CEMEX,AC
Exchange Date,,,,,,,,,
2021-09-24,0.005888,-0.001734,0.010050,-0.010686,0.008265,-0.016164,0.018993,0.011704,-0.005653
2021-10-01,-0.029063,-0.034241,-0.039883,-0.041529,-0.060640,-0.036752,-0.029027,-0.027001,-0.013306
2021-10-08,0.001753,0.019889,0.023747,0.001633,0.004291,-0.009330,-0.035120,-0.083915,-0.001885
2021-10-15,0.013576,0.031745,0.011321,0.036611,0.049494,0.027540,0.014987,0.100711,0.029169
2021-10-22,0.026581,0.016272,-0.026891,-0.021552,0.039521,0.043596,-0.005385,-0.067203,-0.006676


Independent Variables : Market Features


,US_Market_Returns,MX_Market_Returns,Market_Correlation
Exchange Date,,,
2021-09-24,0.005073,-0.005224,NaN
2021-10-01,-0.022094,-0.019918,NaN
2021-10-08,0.007872,-0.011086,NaN
2021-10-15,0.018225,0.050781,NaN
2021-10-22,0.016445,-0.009218,NaN


S&P      0.022591
IPC      0.031559
AAPL     0.038274
MSFT     0.038827
GOOGL    0.045004
dtype: float64